# Module 05: Neural Networks from Scratch
## Chapter 5.6 — Gradient Descent & Optimizers: The Acceleration Engines of AI

> **"Gradients tell you which way is downhill; the optimizer decides how fast to run, how to dodge obstacles, and how to avoid launching yourself off the cliff."**

---

### Executive Overview & Learning Path

In **Chapter 5.5**, we built a backpropagation engine capable of computing exact analytical gradients $\nabla_{\mathbf{W}} J$ for every parameter in a network. But knowing the gradient direction is only half the battle.

How should we step downhill? 
- If our step size is too small, training will take a century.
- If our step size is too large, the network will oscillate wildly and explode to `NaN`.
- If our loss landscape has narrow ravines, standard gradient descent will violently bounce wall-to-wall while barely inching forward.

In this capstone chapter of **Module 05**, we trace the evolution of numerical optimization algorithms from classical **Stochastic Gradient Descent (SGD)**, through physics-inspired **Momentum**, to adaptive learning rate engines like **RMSprop** and **Adam (Adaptive Moment Estimation)**. We conclude by applying our pure NumPy optimizers to train a deep neural network on the full **MNIST handwritten digits dataset**, achieving $> 95\%$ test accuracy with zero deep learning frameworks!

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                               CHAPTER 5.6 ROADMAP                                      │
├────────────────────────────────────────────────────────────────────────────────────────┤
│ 1. THE GD SPECTRUM           ──> Batch vs Stochastic vs Mini-Batch: Speed vs Variance  │
│ 2. MOMENTUM & RAVINES        ──> The Bowling Ball Physics of Escaping Saddle Points    │
│ 3. ADAPTIVE LEARNING RATES   ──> AdaGrad, RMSprop, and Moving Averages of Gradients    │
│ 4. THE ADAM OPTIMIZER        ──> 1st Moment + 2nd Moment + Bias Correction Explained   │
│ 5. LEARNING RATE SCHEDULES   ──> Step Decay, Linear Warmup, and Cosine Annealing       │
│ 6. CAPSTONE MNIST BENCHMARK  ──> Training a Pure NumPy Neural Net to >95% Test Accuracy │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

---


---

## 1. The Gradient Descent Spectrum — Batch vs Stochastic vs Mini-Batch

---

### 🗣️ In Simple English

How many training samples should you look at before updating your network's weights?

Imagine you are surveying voters before an election:
1. **Batch Gradient Descent (The Census):** You interview **every single citizen in the country** ($100\%$ of data) before publishing an update. Your estimate is rock-solid and perfectly accurate, but the survey takes 6 months and costs billions of dollars.
2. **Stochastic Gradient Descent / SGD (The Man on the Street):** You walk outside, interview **one single random pedestrian**, and immediately declare the winner. It takes 10 seconds, but if you happen to bump into an eccentric billionaire, your election prediction swings wildly into chaos!
3. **Mini-Batch Gradient Descent (The Focus Group):** You interview a carefully sampled group of **64 or 128 voters**. It is fast, cheap, and representative enough to make high-confidence decisions.

The key point is: **Mini-Batch Gradient Descent ($B = 32 \dots 512$) is the universal practical default of deep learning**. It matches the parallel matrix multiplication hardware of modern GPUs while introducing just enough random noise to shake the model out of shallow local minima!

---

### 🌍 Real-Life Example

**Scenario:** Training an image classifier on 1,000,000 high-resolution photos.

| Optimization Regime | Batch Size ($B$) | Updates per Epoch | Compute Hardware Behavior | Trajectory Dynamics |
|:---|:---:|:---:|:---|:---|
| **Batch GD** | $1,000,000$ (All) | Exactly $1$ | Exceeds GPU memory (OOM Crash); immense latency per step | Completely smooth, deterministic, but easily trapped in sharp local minima |
| **Stochastic GD (Pure SGD)** | $1$ (Single photo) | $1,000,000$ | Underutilizes GPU tensor cores; poor SIMD throughput | Extreme high-frequency oscillations; noisy wandering |
| **Mini-Batch GD** | $128$ (Standard) | $7,812$ | **100% GPU Tensor Core Saturation**; maximum throughput | Gentle stochastic bounce; naturally finds flat, generalizable minima |

**Why this matters:** GPUs are designed to multiply matrices in chunks of powers of two ($32, 64, 128, 256$). Mini-batching processes 128 images almost as fast as a single image!

---

### 📖 How It Works (Step by Step)

#### A. Batch Gradient Descent
1. Ingest entire training dataset: $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$.
2. Compute average gradient across all $N$ samples:
   $$\mathbf{g} = \frac{1}{N} \sum_{i=1}^N \nabla_\theta L(\mathbf{x}_i, y_i; \theta)$$
3. Update parameters once per epoch:
   $$\theta \leftarrow \theta - \eta \cdot \mathbf{g}$$

#### B. Stochastic Gradient Descent (Pure SGD)
1. Shuffle dataset randomly.
2. For each individual sample $i \in \{1, \dots, N\}$:
   $$\mathbf{g}_i = \nabla_\theta L(\mathbf{x}_i, y_i; \theta)$$
   $$\theta \leftarrow \theta - \eta \cdot \mathbf{g}_i$$
   *(Updates parameters $N$ times per epoch).*

#### C. Mini-Batch Gradient Descent
1. Shuffle dataset and partition into mini-batches of size $B$ (e.g., $B = 64$).
2. For each batch $\mathcal{B} = \{(\mathbf{x}_k, y_k)\}_{k=1}^B$:
   $$\mathbf{g}_{\mathcal{B}} = \frac{1}{B} \sum_{k=1}^B \nabla_\theta L(\mathbf{x}_k, y_k; \theta)$$
   $$\theta \leftarrow \theta - \eta \cdot \mathbf{g}_{\mathcal{B}}$$

---

### 🔑 Key Concepts

| Mode | Batch Size ($B$) | Gradient Variance | GPU Efficiency | Convergence Path |
|:---|:---:|:---:|:---:|:---|
| **Batch GD** | $N$ (Full data) | $0$ (Deterministic) | Poor (Memory limited) | Direct, smooth line |
| **SGD** | $1$ | High (Noisy) | Very Low (Memory bound) | Jagged random walk |
| **Mini-Batch GD** | $32 \dots 256$ | Controlled | Optimal (Tensor Cores) | Smooth trajectory with gentle exploration |

---

### 🗺️ Visual Overview

```
OPTIMIZATION PATHS TOWARD THE MINIMUM ON A 2D CONTOUR:

Weight w2
  ^
  |        /-----------------------------------\
  |       /   Batch GD (Smooth, direct line)    \
  |      /   =============================> ( * ) \  <-- Global Minimum
  |     /                                          \
  |    |     SGD (Wild noisy jagged zig-zags)       |
  |    |     /\    /\    /\   /\                    |
  |    |    /  \  /  \  /  \ /  \=======> ( * )     |
  |     \   `--'  `--'  `--'                       /
  |      \   Mini-Batch GD (Gentle guided curve)  /
  |       \   ~.~.~.~.~.~.~.~.~.~.~.~.~.~> ( * ) /
  +----------------------------------------------------> Weight w1
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Batch Gradient Descent is always superior because its gradients have zero noise."* | **A common beginner fallacy!** In deep non-convex landscapes, zero noise is actually a **curse**. Batch GD gets easily trapped in shallow, sharp local minima that fail to generalize to unseen test data. The mild stochastic noise in Mini-Batch GD acts as a regularizer, helping the model bounce out of sharp ravines and settle into broad, flat basins! |

---

### ✅ When to Use This

- Always use **Mini-Batch Gradient Descent** ($B = 32, 64, 128$) for neural network training.
- Choose $B$ based on GPU memory capacity: maximize $B$ until tensor cores are saturated.

---

### 📊 Section Quick Reference

| Parameter | Batch GD | Mini-Batch GD | Pure SGD |
|:---|:---:|:---:|:---:|
| **Sample Size per Step** | $N$ | $B$ (typically $64$) | $1$ |
| **Step Direction Accuracy** | $100\%$ True Gradient | Unbiased Expectation $\mathbb{E}[\mathbf{g}_{\mathcal{B}}] = \nabla J$ | High Variance Estimate |
| **Memory Requirement** | $O(N)$ | $O(B)$ | $O(1)$ |

---


In [1]:
# ✅ Code Demo: Simulating Batch GD vs Pure SGD vs Mini-Batch GD Trajectories
# Concept: Gradient Variance, Optimization Paths, and Convergence Speeds on 2D Bowl
# ---------------------------------------------------------------------------------

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Define Anisotropic 2D Quadratic Loss: J(w1, w2) = 0.5 * (w1^2 + 10 * w2^2)
# Steeper along w2 axis than w1 axis (simulating real ill-conditioned valleys)
def loss_func(w1, w2):
    return 0.5 * (w1**2 + 10.0 * w2**2)

def true_gradient(w1, w2):
    return np.array([w1, 10.0 * w2])

# Generate synthetic dataset of 200 noisy gradient observations
np.random.seed(42)
N_data = 200
noise_scale = 4.0

def get_sample_gradient(w1, w2, batch_indices):
    base_grad = true_gradient(w1, w2)
    # Add sample noise reflecting individual data points
    noise = np.random.randn(len(batch_indices), 2) * noise_scale
    sample_grads = base_grad + noise
    return np.mean(sample_grads, axis=0)

# 2. Run Optimization from starting point (w1=4.0, w2=2.0)
start_pos = np.array([4.0, 2.0])
lr = 0.1
n_steps = 35

# Mode A: Batch GD (Uses all 200 samples)
path_batch = [start_pos.copy()]
curr = start_pos.copy()
for _ in range(n_steps):
    g = get_sample_gradient(curr[0], curr[1], range(N_data))
    curr -= lr * g
    path_batch.append(curr.copy())

# Mode B: Mini-Batch GD (Batch size = 16)
path_minibatch = [start_pos.copy()]
curr = start_pos.copy()
for _ in range(n_steps):
    idx = np.random.choice(N_data, size=16, replace=False)
    g = get_sample_gradient(curr[0], curr[1], idx)
    curr -= lr * g
    path_minibatch.append(curr.copy())

# Mode C: Pure SGD (Batch size = 1)
path_sgd = [start_pos.copy()]
curr = start_pos.copy()
for _ in range(n_steps):
    idx = np.random.choice(N_data, size=1, replace=False)
    g = get_sample_gradient(curr[0], curr[1], idx)
    curr -= lr * g
    path_sgd.append(curr.copy())

path_batch = np.array(path_batch)
path_minibatch = np.array(path_minibatch)
path_sgd = np.array(path_sgd)

# 3. Visualize Trajectories on Contour Surface
w1_grid = np.linspace(-4.5, 4.5, 150)
w2_grid = np.linspace(-2.5, 2.5, 150)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
Z_loss = loss_func(W1, W2)

plt.figure(figsize=(10, 6.5))
plt.contourf(W1, W2, Z_loss, levels=25, cmap='Blues_r', alpha=0.75)
plt.contour(W1, W2, Z_loss, levels=15, colors='white', alpha=0.3, linewidths=0.8)

plt.plot(path_batch[:, 0], path_batch[:, 1], color='#ef4444', marker='o', markersize=4, lw=2.2, label='Batch GD (B=200): Direct & Smooth')
plt.plot(path_minibatch[:, 0], path_minibatch[:, 1], color='#10b981', marker='s', markersize=4, lw=2.0, label='Mini-Batch GD (B=16): Stable with slight exploration')
plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='#f59e0b', marker='^', markersize=4, lw=1.5, linestyle='--', label='Pure SGD (B=1): Noisy & High Variance')

plt.scatter([0], [0], color='#dc2626', s=160, marker='*', zorder=6, label='Global Minimum (0, 0)')
plt.scatter([start_pos[0]], [start_pos[1]], color='black', s=120, zorder=6, label='Start Point (4, 2)')

plt.title('Optimization Trajectories: Batch vs Mini-Batch vs Pure SGD', fontsize=12, fontweight='bold')
plt.xlabel('Parameter w1')
plt.ylabel('Parameter w2')
plt.xlim(-4.5, 4.5)
plt.ylim(-2.5, 2.5)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower left', frameon=True, facecolor='white')
plt.tight_layout()

fig_path = os.path.join(assets_dir, 'gd_variants_trajectories.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Figure successfully saved to: {fig_path}")

print("\nConvergence Summary (Final Distance to Optimum (0, 0)):")
print(f"  Batch GD (B=200):      Distance = {np.linalg.norm(path_batch[-1]):.4f}")
print(f"  Mini-Batch GD (B=16):  Distance = {np.linalg.norm(path_minibatch[-1]):.4f}")
print(f"  Pure SGD (B=1):        Distance = {np.linalg.norm(path_sgd[-1]):.4f} (Oscillates around minimum)")


Figure successfully saved to: Module-05-Neural-Networks/assets\gd_variants_trajectories.png

Convergence Summary (Final Distance to Optimum (0, 0)):
  Batch GD (B=200):      Distance = 0.0480
  Mini-Batch GD (B=16):  Distance = 0.2884
  Pure SGD (B=1):        Distance = 1.5617 (Oscillates around minimum)


---

## 2. Escaping Ravines & Saddles — Classical Momentum & Nesterov Accelerated Gradient

---

### 🗣️ In Simple English

Have you ever tried rolling a heavy bowling ball down a bumpy, uneven grass hill?

If the bowling ball hits a tiny pebble or a small pothole, it doesn't instantly stop or veer off course. Why? Because **it has momentum ($p = mv$)**! Its heavy physical velocity carries it straight through small bumps, and as it rolls downhill, it builds up serious forward speed.

Now, imagine what standard gradient descent does in a steep mountain ravine:
- The walls of the canyon are steep, but the canyon floor slopes downward very gently.
- Standard SGD has zero memory. It looks at the steep walls and leaps across the ravine. Then it looks at the opposite wall and leaps back. It spends $95\%$ of its energy bouncing back and forth across the canyon walls, making almost zero forward progress down the riverbed!

**Momentum** solves this by adding a **velocity vector ($v$)**. Oscillations across the canyon walls point in opposite directions, so they **cancel each other out** ($+1 - 1 = 0$). Meanwhile, the tiny forward steps down the ravine point in the same direction, so they **compound and accelerate**!

---

### 🌍 Real-Life Example

**Scenario:** A bobsled accelerating down an Olympic ice track.

| Physical Component | Momentum Algorithm Equivalent | Mathematical Function |
|:---|:---|:---|
| **Gravity Pulling Downhill** | Current Mini-batch Gradient ($\mathbf{g}_t$) | The instantaneous slope vector |
| **Bobsled Mass & Velocity** | Velocity Vector ($\mathbf{v}_t$) | Exponential moving average of past gradients |
| **Ice Friction Coefficient** | Momentum Hyperparameter ($\beta \approx 0.9$) | Dampening factor ($0.9$ retains $90\%$ of past speed) |
| **Bobsled Steering Ahead** | **Nesterov Accelerated Gradient (NAG)** | Evaluating the gradient at the anticipated future spot before turning |

**Why this matters:** In deep neural networks with billions of weights, the loss surface is full of narrow ravines (ill-conditioned curvature) and flat plateaus (saddle points). Momentum accelerates learning by up to $10\times$ and powers the network straight across saddle points where the gradient is temporarily zero!

---

### 📖 How It Works (Step by Step)

#### A. Classical Polyak Momentum (1964)
1. **Initialize Velocity:** $\mathbf{v}_0 = 0$.
2. **Compute Mini-Batch Gradient:** $\mathbf{g}_t = \nabla_\theta J(\theta_t)$.
3. **Update Velocity Vector:**
   $$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \eta \mathbf{g}_t$$
   *(where $\beta \in [0.8, 0.99]$, standard default $\beta = 0.9$).*
4. **Update Parameters:**
   $$\theta_t \leftarrow \theta_{t-1} - \mathbf{v}_t$$

*Alternative Formulation (Moving Average):*
$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + (1 - \beta) \mathbf{g}_t, \quad \theta_t \leftarrow \theta_{t-1} - \eta \mathbf{v}_t$$
Both formulations are mathematically equivalent up to scaling of $\eta$.

#### B. Nesterov Accelerated Gradient (NAG) (1983)
- In standard momentum, we compute the gradient at our current position $\theta_{t-1}$, then add momentum.
- Yurii Nesterov realized: *"We already know that momentum is going to carry us in direction $\beta \mathbf{v}_{t-1}$. Why not look ahead and evaluate the gradient at that future position?"*
1. **Look-Ahead Step:** $\theta_{\text{ahead}} = \theta_{t-1} - \beta \mathbf{v}_{t-1}$.
2. **Evaluate Look-Ahead Gradient:** $\mathbf{g}_t = \nabla_\theta J(\theta_{\text{ahead}})$.
3. **Update Velocity & Parameters:**
   $$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \eta \mathbf{g}_t, \quad \theta_t \leftarrow \theta_{t-1} - \mathbf{v}_t$$
- **Advantage:** Acts like a smart braking system. If the velocity is about to carry the model up a steep hill, the look-ahead gradient detects the uphill slope early and applies the brakes!

---

### 🔑 Key Concepts

| Optimizer | Memory Vector | Effective Speedup in Constant Gradient | Primary Strength |
|:---|:---:|:---:|:---|
| **Vanilla SGD** | None | $1.0\times$ | Simple, baseline |
| **SGD with Momentum** | Velocity $\mathbf{v}_t$ | $\frac{1}{1 - \beta} \approx 10\times$ (for $\beta=0.9$) | Dampens orthogonal oscillations, accelerates down ravines |
| **Nesterov (NAG)** | Look-ahead $\mathbf{v}_t$ | $\approx 10\times$ | Prevents overshooting at sharp valley turns |

---

### 🗺️ Visual Overview

```
MOMENTUM CANCELLING OSCILLATIONS IN A STEEP RAVINE:

Vanilla SGD: Bounces back and forth between steep walls
Wall A  |    \      /      \      /      \
        |     \    /        \    /        \
        |      \  /          \  /          \
Wall B  |       \/            \/            \=========> Inefficient crawl!

SGD + Momentum: Perpendicular bounces cancel (+ - = 0), forward momentum adds up!
Wall A  |
        |    =========================================> Blazing fast forward rocket!
Wall B  |
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"If momentum is good, setting $\beta = 0.999$ must be even better."* | **Danger!** If $\beta$ is set too close to $1.0$, the bowling ball has too much inertia. It will overshoot the minimum, climb up the opposite canyon wall, and oscillate wildly before coming to rest. $\beta = 0.9$ is the universally tuned sweet spot. |

---

### ✅ When to Use This

- Always add Momentum ($\beta = 0.9$) whenever training with SGD.
- Standard default optimizer for training deep **ResNet vision architectures**.

---

### 📊 Section Quick Reference

| Parameter | Symbol | Standard Value | Role |
|:---|:---:|:---:|:---|
| **Momentum Coefficient** | $\beta$ | $0.9$ | Memory retention factor |
| **Velocity Vector** | $\mathbf{v}$ | Initialized to $0$ | Running directional momentum |
| **Speedup Factor** | $\frac{1}{1 - \beta}$ | $10\times$ | Effective step multiplier along consistent gradients |

---


In [2]:
# ✅ Code Demo: Vanilla SGD vs Momentum in an Ill-Conditioned Steep Ravine
# Concept: Ravine Oscillations, Velocity Vector Accumulation, and Overshoot Damping
# ---------------------------------------------------------------------------------

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Define Steep Ravine: J(w1, w2) = 0.5 * (w1^2 + 50 * w2^2)
# The w2 axis is 50x steeper than the w1 axis!
def ravine_loss(w1, w2):
    return 0.5 * (w1**2 + 50.0 * w2**2)

def ravine_grad(w1, w2):
    return np.array([w1, 50.0 * w2])

start = np.array([4.0, 1.5])
lr = 0.035
n_steps = 40

# A. Vanilla SGD
path_sgd = [start.copy()]
curr_sgd = start.copy()
for _ in range(n_steps):
    g = ravine_grad(curr_sgd[0], curr_sgd[1])
    curr_sgd -= lr * g
    path_sgd.append(curr_sgd.copy())

# B. SGD with Momentum (beta = 0.85)
path_mom = [start.copy()]
curr_mom = start.copy()
v = np.zeros(2)
beta = 0.85
for _ in range(n_steps):
    g = ravine_grad(curr_mom[0], curr_mom[1])
    v = beta * v + lr * g
    curr_mom -= v
    path_mom.append(curr_mom.copy())

path_sgd = np.array(path_sgd)
path_mom = np.array(path_mom)

# 2. Visualize Trajectories on the Steep Ravine
w1_vals = np.linspace(-4.5, 4.5, 200)
w2_vals = np.linspace(-2.0, 2.0, 200)
W1, W2 = np.meshgrid(w1_vals, w2_vals)
Z_ravine = ravine_loss(W1, W2)

plt.figure(figsize=(11, 5.5))
plt.contourf(W1, W2, Z_ravine, levels=30, cmap='Purples_r', alpha=0.8)
plt.contour(W1, W2, Z_ravine, levels=15, colors='white', alpha=0.25, linewidths=0.8)

plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='#ef4444', marker='o', markersize=3.5, lw=1.8, label='Vanilla SGD: Severe vertical oscillations')
plt.plot(path_mom[:, 0], path_mom[:, 1], color='#10b981', marker='s', markersize=4, lw=2.2, label=f'Momentum (beta={beta}): Smooth accelerated path')

plt.scatter([0], [0], color='#fbbf24', s=160, marker='*', zorder=6, label='Global Minimum (0, 0)')
plt.scatter([start[0]], [start[1]], color='black', s=120, zorder=6, label='Start Point (4, 1.5)')

plt.title('The Ravine Problem: Vanilla SGD vs Momentum (50x Condition Ratio)', fontsize=12, fontweight='bold')
plt.xlabel('Parameter w1 (Gentle Slope)')
plt.ylabel('Parameter w2 (Steep Canyon Wall)')
plt.xlim(-4.5, 4.5)
plt.ylim(-2.0, 2.0)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower left', frameon=True, facecolor='white')
plt.tight_layout()

fig_path = os.path.join(assets_dir, 'momentum_ravine_escape.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Figure successfully saved to: {fig_path}")

print("Ravine Benchmark Summary (Steps = 40):")
print(f"  Vanilla SGD Final Cost: {ravine_loss(path_sgd[-1,0], path_sgd[-1,1]):.6f} (Trapped bouncing across w2)")
print(f"  Momentum Final Cost:    {ravine_loss(path_mom[-1,0], path_mom[-1,1]):.6f} (Accelerated cleanly to minimum)")


Figure successfully saved to: Module-05-Neural-Networks/assets\momentum_ravine_escape.png
Ravine Benchmark Summary (Steps = 40):
  Vanilla SGD Final Cost: 0.462674 (Trapped bouncing across w2)
  Momentum Final Cost:    0.012737 (Accelerated cleanly to minimum)


---

## 3. Adaptive Learning Rates — AdaGrad and RMSprop

---

### 🗣️ In Simple English

Why should every parameter in a 100-million-parameter neural network share the **exact same learning rate**?

Think of it this way: Imagine a basketball team:
- The center is $215\text{ cm}$ tall and takes huge, heavy steps.
- The point guard is fast, agile, and takes rapid, tiny steps.
- If the coach shouts: *"Everyone must take exactly $1.5\text{-meter}$ strides!"*, the point guard will trip and fall, while the center will feel unnaturally restricted.

In a neural network, some weights receive massive gradient updates every step, while others (like weights connected to rare vocabulary words) receive tiny gradients once a week. 
- **Adaptive learning rate algorithms** assign a **custom, personalized step size to every single individual weight** in the network!
- If a weight's gradients have been huge and violent, the optimizer **shrinks** its learning rate to keep it stable.
- If a weight's gradients have been small and quiet, the optimizer **boosts** its learning rate so it makes meaningful progress!

---

### 🌍 Real-Life Example

**Scenario:** Training a natural language model on English text containing common words (*"the"*, *"is"*) and rare medical terms (*"pseudopseudohypoparathyroidism"*).

| Token Frequency | Gradient Frequency | Problem with Fixed $\eta$ | How RMSprop Solves It |
|:---|:---|:---|:---|
| Frequent: *"the"* | Gradient fires on every single sample; gradient norm is huge | Prone to overshooting and exploding weights | **Divides by $\sqrt{s}$:** Dampens learning rate to maintain delicate convergence |
| Rare: *"pseudohypo..."* | Gradient is zero $99.99\%$ of the time | Parameter barely moves; network never learns the word | **Divides by tiny $\sqrt{s}$:** Boosts effective learning rate, accelerating learning of rare features |

---

### 📖 How It Works (Step by Step)

#### A. AdaGrad (Duchi et al., 2011) — The Pioneer
1. Accumulate the sum of squared historical gradients:
   $$G_t = G_{t-1} + \mathbf{g}_t^2$$
2. Update weights by scaling inversely with $\sqrt{G_t}$:
   $$\theta_t \leftarrow \theta_{t-1} - \frac{\eta}{\sqrt{G_t + \epsilon}} \odot \mathbf{g}_t$$
- **The Fatal Flaw of AdaGrad:** Because $g_t^2 \ge 0$, the sum $G_t$ **strictly increases on every single step** without bound. As $t \to \infty$, $G_t \to \infty \implies \frac{\eta}{\sqrt{G_t}} \to 0$. The learning rate evaporates to zero, permanently freezing the network before training finishes!

#### B. RMSprop (Geoffrey Hinton, Coursera Lecture 6, 2012) — The Solution
Hinton fixed AdaGrad by replacing the unbounded cumulative sum with an **Exponential Moving Average (EMA)** of squared gradients. It only remembers recent gradients!
1. **Compute Squared Gradient EMA:**
   $$s_t = \beta s_{t-1} + (1 - \beta) \mathbf{g}_t^2$$
   *(Standard default $\beta = 0.99$ or $0.9$, $\epsilon = 10^{-8}$).*
2. **Update Parameters:**
   $$\theta_t \leftarrow \theta_{t-1} - \frac{\eta}{\sqrt{s_t + \epsilon}} \odot \mathbf{g}_t$$
- **Why this works:** $s_t$ does not grow monotonically. If recent gradients shrink, $s_t$ shrinks, allowing the effective learning rate to automatically increase!

---

### 🔑 Key Concepts

| Optimizer | Second-Moment Accumulator ($s_t$) | Solves Premature Freezing? | Key Innovation |
|:---|:---|:---:|:---|
| **AdaGrad** | Cumulative sum: $\sum \mathbf{g}_t^2$ | ❌ No ($s_t \to \infty$) | First per-parameter adaptive learning rate |
| **RMSprop** | Exponential Moving Average: $\beta s_{t-1} + (1-\beta)\mathbf{g}_t^2$ | ✅ Yes | Leaky memory prevents learning rate collapse |

---

### 🗺️ Visual Overview

```
ADAPTIVE LEARNING RATE MECHANISM:

Gradient Magnitude g_t ──> [ Square g_t^2 ] ──> [ EMA: s_t = beta*s_{t-1} + (1-beta)*g^2 ]
                                                                      │
                                                                      ▼
Parameter Update:  Delta_theta = - ( eta / sqrt(s_t + eps) ) * g_t
                   ▲
                   Large recent gradients  --> Large s_t --> Smaller effective step!
                   Small recent gradients  --> Small s_t --> Larger effective step!
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"RMSprop was published in a prestigious NeurIPS or ICML research paper."* | **A famous piece of AI folklore!** Geoffrey Hinton never published RMSprop in an academic paper. He proposed it casually on Slide 29 of Lecture 6 in his free 2012 Coursera online deep learning course! It worked so phenomenally well that the entire global research community adopted it immediately. |

---

### ✅ When to Use This

- **Use RMSprop:** Especially popular in Recurrent Neural Networks (RNNs) and Reinforcement Learning (Deep Q-Networks / DQN).

---

### 📊 Section Quick Reference

| Hyperparameter | Symbol | Typical Default | Purpose |
|:---|:---:|:---:|:---|
| **Moving Average Decay** | $\beta$ | $0.9$ or $0.99$ | Controls memory window of past gradient scale |
| **Stability Constant** | $\epsilon$ | $10^{-8}$ | Prevents division by zero when gradient is zero |
| **Base Learning Rate** | $\eta$ | $0.001$ | Baseline step size |

---


---

## 4. The King of Optimizers — Adam (Adaptive Moment Estimation)

---

### 🗣️ In Simple English

If you could take the best features of **Momentum** (speed, bowling ball physics, escaping ravines) and merge them with the best features of **RMSprop** (adaptive per-parameter step sizes, handling sparse features), what would you get?

You get **Adam (Adaptive Moment Estimation)**, published by Diederik Kingma and Jimmy Ba in 2014.

Think of it this way:
- **Momentum** controls the **Steering Wheel and Accelerator**: it keeps you moving smoothly in the right direction.
- **RMSprop** controls the **Automatic Suspension**: it cushions the bumps when the road is rough and lowers the car when the highway is flat.
- **Adam is the Tesla Autopilot** combining both systems simultaneously!

Because Adam works out-of-the-box on $95\%$ of deep learning tasks without needing painful hyperparameter tuning, it is the **undisputed champion optimizer of modern AI**.

---

### 🌍 Real-Life Example

**Scenario:** Training modern foundation models (GPT-4, LLaMA-3, Stable Diffusion).

Virtually every single state-of-the-art Large Language Model and Generative AI system in existence today is trained using **Adam** (or its weight-decay variant, **AdamW**)!

| Parameter Component in Adam | Role in LLM Pre-Training |
|:---|:---|
| First Moment ($m_t$) | Preserves directional velocity across long token sequences |
| Second Moment ($v_t$) | Balances gradient scales between dense attention projections and sparse embedding tables |
| Bias Correction ($\hat{m}_t, \hat{v}_t$) | Prevents catastrophic gradient explosions during the crucial first 1,000 steps of training |

---

### 📖 How It Works (Step by Step)

Adam tracks two distinct running vectors for every parameter: the **first moment** (mean gradient direction) and the **second moment** (un-centered gradient variance).

#### Step 1: Update Biased First Moment (Momentum)
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) \mathbf{g}_t$$
*(Standard default $\beta_1 = 0.9$).*

#### Step 2: Update Biased Second Moment (RMSprop)
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) \mathbf{g}_t^2$$
*(Standard default $\beta_2 = 0.999$).*

#### Step 3: Bias Correction (The Crucial Fix!)
At step $t = 0$, both vectors are initialized to zero ($m_0 = 0, v_0 = 0$).
Because $\beta_1$ and $\beta_2$ are close to $1.0$ (e.g., $\beta_2 = 0.999$), on early steps $m_t$ and $v_t$ are **heavily biased toward zero**!
For example, at $t = 1$:
$$v_1 = 0.999(0) + 0.001 \mathbf{g}_1^2 = 0.001 \mathbf{g}_1^2$$
Notice that $v_1$ is 1,000 times too small! If we divided by $\sqrt{v_1}$, the learning rate would explode by $\sqrt{1000} \approx 31.6\times$!

To fix this, Kingma and Ba proved that dividing by $(1 - \beta^t)$ yields the exact mathematically unbiased expectation:
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
- Notice that as $t \to \infty$, $\beta^t \to 0 \implies (1 - \beta^t) \to 1.0$. The correction naturally fades away as training matures!

#### Step 4: Parameter Update
$$\theta_t \leftarrow \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

---

### 🔑 Key Concepts

| Hyperparameter | Recommended Default | Role |
|:---:|:---:|:---|
| $\eta$ (Learning Rate) | $0.001$ ($10^{-3}$) | Base step size scale |
| $\beta_1$ | $0.9$ | First moment decay (momentum) |
| $\beta_2$ | $0.999$ | Second moment decay (RMSprop) |
| $\epsilon$ | $10^{-8}$ | Numerical stability floor |
| $1 - \beta^t$ | Time-dependent scalar | Cold-start zero-bias correction factor |

---

### 🗺️ Visual Overview

```
THE COMPLETE ADAM OPTIMIZER COMPUTATIONAL PIPELINE:

                         Mini-Batch Gradient g_t
                                    │
           ┌────────────────────────┴────────────────────────┐
           ▼                                                 ▼
[ 1st Moment: Momentum ]                          [ 2nd Moment: RMSprop ]
 m_t = beta1*m + (1-beta1)*g                       v_t = beta2*v + (1-beta2)*g^2
           │                                                 │
           ▼                                                 ▼
[ Bias Correction: 1 - beta1^t ]                  [ Bias Correction: 1 - beta2^t ]
 m_hat = m_t / (1 - beta1^t)                       v_hat = v_t / (1 - beta2^t)
           │                                                 │
           └────────────────────────┬────────────────────────┘
                                    ▼
                Update:  theta = theta - eta * ( m_hat / (sqrt(v_hat) + eps) )
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Adam is always strictly superior to SGD with Momentum for all tasks."* | **Not always!** While Adam converges $3\times$ to $5\times$ faster and is much easier to tune, properly tuned **SGD with Momentum and Cosine Annealing** often achieves slightly higher final test accuracy ($+0.5\%$ to $+1.0\%$) on classic computer vision benchmarks (like ImageNet with ResNet) due to subtle generalization differences. |

---

### ✅ When to Use This

- **Use Adam as your first choice** on every new deep learning project, Transformer, NLP task, and generative model.

---

### 📊 Section Quick Reference

| Feature | Adam |
|:---|:---|
| **Formula** | $\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{v}} + \epsilon} \hat{m}$ |
| **Memory Overhead** | $2\times$ model parameters (stores both $m$ and $v$ tensors in GPU RAM) |
| **Typical Learning Rate** | $10^{-3}$ ($0.001$) for MLPs/CNNs; $10^{-4}$ to $10^{-5}$ for Transformers |

---


In [3]:
# ✅ Code Demo: Object-Oriented Modular Pure NumPy Optimizers Engine
# Concept: SGD, Momentum, RMSprop, and Adam Implemented from Mathematical Update Rules
# ------------------------------------------------------------------------------------

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Base Optimizer Interface
class BaseOptimizer:
    def __init__(self, lr=0.01):
        self.lr = lr
    def update(self, param, grad, key):
        raise NotImplementedError

# 2. Vanilla SGD
class VanillaSGD(BaseOptimizer):
    def update(self, param, grad, key):
        return param - self.lr * grad

# 3. SGD with Momentum
class SGDMomentum(BaseOptimizer):
    def __init__(self, lr=0.01, beta=0.9):
        super().__init__(lr)
        self.beta = beta
        self.v = {}
    def update(self, param, grad, key):
        if key not in self.v:
            self.v[key] = np.zeros_like(param)
        self.v[key] = self.beta * self.v[key] + self.lr * grad
        return param - self.v[key]

# 4. RMSprop
class RMSprop(BaseOptimizer):
    def __init__(self, lr=0.01, beta=0.9, eps=1e-8):
        super().__init__(lr)
        self.beta = beta
        self.eps = eps
        self.s = {}
    def update(self, param, grad, key):
        if key not in self.s:
            self.s[key] = np.zeros_like(param)
        self.s[key] = self.beta * self.s[key] + (1.0 - self.beta) * (grad ** 2)
        return param - (self.lr / (np.sqrt(self.s[key]) + self.eps)) * grad

# 5. Adam (Adaptive Moment Estimation)
class Adam(BaseOptimizer):
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = {}
        self.v = {}
        self.t = 0
    def update(self, param, grad, key):
        if key not in self.m:
            self.m[key] = np.zeros_like(param)
            self.v[key] = np.zeros_like(param)
        
        # Increment step on first parameter update per cycle
        if key == 'param_0':
            self.t += 1
            
        t = max(self.t, 1)
        # 1st moment (momentum)
        self.m[key] = self.beta1 * self.m[key] + (1.0 - self.beta1) * grad
        # 2nd moment (RMSprop)
        self.v[key] = self.beta2 * self.v[key] + (1.0 - self.beta2) * (grad ** 2)
        
        # Bias corrections
        m_hat = self.m[key] / (1.0 - self.beta1 ** t)
        v_hat = self.v[key] / (1.0 - self.beta2 ** t)
        
        return param - (self.lr / (np.sqrt(v_hat) + self.eps)) * m_hat

# 6. Head-to-Head Benchmark on Non-Convex Rastrigin-Style 2D Function
def complex_loss(w1, w2):
    return 0.2 * (w1**2 + w2**2) - np.cos(2*np.pi*w1) - np.cos(2*np.pi*w2) + 2.0

def complex_grad(w1, w2):
    dw1 = 0.4 * w1 + 2*np.pi * np.sin(2*np.pi*w1)
    dw2 = 0.4 * w2 + 2*np.pi * np.sin(2*np.pi*w2)
    return np.array([dw1, dw2])

start_point = np.array([1.8, 1.8])
n_iters = 60

optimizers = {
    'Vanilla SGD (lr=0.02)': VanillaSGD(lr=0.02),
    'Momentum (beta=0.85)': SGDMomentum(lr=0.02, beta=0.85),
    'RMSprop (beta=0.9)': RMSprop(lr=0.05, beta=0.9),
    'Adam (lr=0.08)': Adam(lr=0.08)
}

trajectories = {}
for name, opt in optimizers.items():
    curr = start_point.copy()
    path = [curr.copy()]
    for _ in range(n_iters):
        g = complex_grad(curr[0], curr[1])
        curr = opt.update(curr, g, key='param_0')
        path.append(curr.copy())
    trajectories[name] = np.array(path)

# 7. Visualize Comparative Optimization Paths
grid_vals = np.linspace(-2.2, 2.2, 200)
W1, W2 = np.meshgrid(grid_vals, grid_vals)
Z_comp = complex_loss(W1, W2)

plt.figure(figsize=(10, 6.5))
plt.contourf(W1, W2, Z_comp, levels=30, cmap='viridis', alpha=0.7)

colors = {'Vanilla SGD (lr=0.02)': '#ef4444', 'Momentum (beta=0.85)': '#f59e0b', 
          'RMSprop (beta=0.9)': '#3b82f6', 'Adam (lr=0.08)': '#10b981'}

for name, path in trajectories.items():
    plt.plot(path[:, 0], path[:, 1], color=colors[name], marker='o', markersize=3, lw=2.0, label=name)

plt.scatter([0], [0], color='#fbbf24', s=180, marker='*', zorder=6, label='Global Minimum (0, 0)')
plt.scatter([start_point[0]], [start_point[1]], color='white', s=120, edgecolors='k', zorder=6, label='Start Point (1.8, 1.8)')

plt.title('Optimizer Showdown: Navigating Complex Non-Convex Landscape', fontsize=12, fontweight='bold')
plt.xlabel('Parameter w1')
plt.ylabel('Parameter w2')
plt.xlim(-2.2, 2.2)
plt.ylim(-2.2, 2.2)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower left', frameon=True, facecolor='white')
plt.tight_layout()

fig_path = os.path.join(assets_dir, 'optimizer_showdown_benchmark.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Figure successfully saved to: {fig_path}")

print("\nFinal Convergence Results (Distance to Global Minimum (0, 0)):")
for name, path in trajectories.items():
    dist = np.linalg.norm(path[-1])
    print(f"  {name:<24} -> Final Distance = {dist:.4f} | Final Cost = {complex_loss(path[-1,0], path[-1,1]):.4f}")


Figure successfully saved to: Module-05-Neural-Networks/assets\optimizer_showdown_benchmark.png

Final Convergence Results (Distance to Global Minimum (0, 0)):
  Vanilla SGD (lr=0.02)    -> Final Distance = 2.8000 | Final Cost = 1.5839
  Momentum (beta=0.85)     -> Final Distance = 2.7986 | Final Cost = 1.5840
  RMSprop (beta=0.9)       -> Final Distance = 2.8297 | Final Cost = 1.6015
  Adam (lr=0.08)           -> Final Distance = 2.8001 | Final Cost = 1.5839


---

## 5. Learning Rate Schedules — The Art of Annealing

---

### 🗣️ In Simple English

Imagine you are trying to park a car into a tight parallel parking spot:
- When you are driving down the street toward the spot, you drive at **$50\text{ km/h}$** (large learning rate).
- When you turn into the parking space, you slow down to **$10\text{ km/h}$**.
- When you are $5\text{ centimeters}$ from the curb, you inch forward at **$1\text{ km/h}$** (tiny learning rate) so you don't scratch the bumper!

If you tried to park while driving at $50\text{ km/h}$, you would crash into the wall! 
In neural networks, **keeping a constant learning rate throughout training is a mistake**. 
- Early in training, weights need large steps to explore and escape initial bad configurations.
- Late in training, weights need tiny, delicate micro-steps to settle into the lowest crevices of the minimum.

---

### 📖 The Three Essential Learning Rate Schedules

#### 1. Step Decay
Drops the learning rate by a fixed factor $\gamma$ (e.g., $0.1$ or $0.5$) every $K$ epochs:
$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / K \rfloor}$$

#### 2. Linear Warmup
When training begins, network weights are random and gradients can be chaotic. Warmup starts with a tiny learning rate and linearly increases it over the first $W$ steps to stabilize training:
$$\eta_t = \eta_{\max} \cdot \frac{t}{W} \quad \text{for } t \le W$$
*(Crucial when training Transformers and LLMs with Adam).*

#### 3. Cosine Annealing (Loshchilov & Hutter, 2016)
Smoothly decays the learning rate following the shape of a half-period cosine curve down to a minimum floor $\eta_{\min}$:
$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t}{T_{\max}} \pi\right)\right)$$
- **Advantage:** No sudden, harsh step drops; smooth continuous decay yields superior test set generalization.

---


In [4]:
# ✅ Code Demo: Visualizing Step Decay, Linear Warmup, and Cosine Annealing Schedules
# Concept: Mathematical Formulation and Trajectory Comparisons over 100 Epochs
# -----------------------------------------------------------------------------

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Schedule Implementations
epochs = 100
base_lr = 0.01
min_lr = 0.0001
warmup_epochs = 10

# Schedule 1: Step Decay (Drops by 0.5 every 25 epochs)
lr_step = [base_lr * (0.5 ** (e // 25)) for e in range(epochs)]

# Schedule 2: Cosine Annealing
lr_cosine = [min_lr + 0.5 * (base_lr - min_lr) * (1.0 + np.cos(np.pi * e / epochs)) for e in range(epochs)]

# Schedule 3: Linear Warmup + Cosine Annealing (Modern LLM Standard)
lr_warmup_cosine = []
for e in range(epochs):
    if e < warmup_epochs:
        lr_val = base_lr * (e + 1) / warmup_epochs
    else:
        progress = (e - warmup_epochs) / (epochs - warmup_epochs)
        lr_val = min_lr + 0.5 * (base_lr - min_lr) * (1.0 + np.cos(np.pi * progress))
    lr_warmup_cosine.append(lr_val)

# 2. Visualize Schedules
plt.figure(figsize=(10, 4.8))
epoch_range = range(1, epochs + 1)

plt.plot(epoch_range, lr_step, color='#ef4444', lw=2.2, linestyle='--', label='Step Decay (Drops 50% every 25 epochs)')
plt.plot(epoch_range, lr_cosine, color='#3b82f6', lw=2.5, label='Cosine Annealing (Smooth half-wave)')
plt.plot(epoch_range, lr_warmup_cosine, color='#10b981', lw=2.5, label='Warmup (10 ep) + Cosine Annealing (SOTA LLM Standard)')

plt.axvspan(1, warmup_epochs, color='#dcfce7', alpha=0.5, label='Warmup Phase (Gradient Stabilization)')
plt.title('Learning Rate Schedules: Step vs Cosine vs Warmup+Cosine', fontsize=12, fontweight='bold')
plt.xlabel('Training Epochs')
plt.ylabel('Learning Rate (eta)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='upper right', frameon=True, facecolor='white')
plt.tight_layout()

fig_path = os.path.join(assets_dir, 'lr_schedules_comparison.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Figure successfully saved to: {fig_path}")


Figure successfully saved to: Module-05-Neural-Networks/assets\lr_schedules_comparison.png


---

## 6. The Capstone Benchmark — Training on Real MNIST Digits from Scratch

---

### 🗣️ In Simple English

We have now built every atomic component of deep learning completely from scratch:
- **Neuron & Layer Architecture** (Chapter 5.1 & 5.2)
- **Non-Linear Activations** (Chapter 5.3)
- **Numerically Stable Loss Functions** (Chapter 5.4)
- **Analytical Matrix Backpropagation** (Chapter 5.5)
- **Mini-Batch Adam Optimizer** (Chapter 5.6)

Now, we bring them all together for the ultimate real-world test: **Recognizing handwritten digits ($0$ through $9$) on the full MNIST dataset** using pure Python and NumPy with zero PyTorch or TensorFlow!

---


In [5]:
# ✅ Code Demo: Full Capstone Training Loop on Real MNIST Handwritten Digits
# Concept: Real Image Normalization, Mini-Batch Adam, and Evaluating Test Set Generalization
# ------------------------------------------------------------------------------------------

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Load Pre-Downloaded MNIST Dataset from Local Workspace
mnist_path = 'Module-05-Neural-Networks/data/mnist.npz' if os.path.exists('Module-05-Neural-Networks/data/mnist.npz') else 'data/mnist.npz'
data = np.load(mnist_path)

x_train_raw = data['x_train']  # Shape: (60000, 28, 28)
y_train_raw = data['y_train']  # Shape: (60000,)
x_test_raw  = data['x_test']   # Shape: (10000, 28, 28)
y_test_raw  = data['y_test']   # Shape: (10000,)

# 2. Data Preprocessing
# Flatten 28x28 images into 784-dimensional feature vectors and normalize pixels to [0, 1]
X_train = x_train_raw.reshape(-1, 784).astype(np.float32) / 255.0
X_test  = x_test_raw.reshape(-1, 784).astype(np.float32) / 255.0

# Subsample 10,000 training examples for rapid interactive training demonstration
np.random.seed(42)
train_subset_idx = np.random.choice(len(X_train), size=10000, replace=False)
X_train_sub = X_train[train_subset_idx]
y_train_sub = y_train_raw[train_subset_idx]

# One-hot encode targets (10 classes: digits 0-9)
C = 10
Y_train_onehot = np.eye(C)[y_train_sub]
Y_test_onehot  = np.eye(C)[y_test_raw]

# 3. Model Architecture: Input(784) -> Hidden(128, ReLU) -> Output(10, Softmax)
class ScratchMNISTModel:
    def __init__(self, in_dim=784, hidden_dim=128, out_dim=10):
        np.random.seed(42)
        # He Normal initialization
        self.W1 = np.random.randn(in_dim, hidden_dim) * np.sqrt(2.0 / in_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, out_dim) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, out_dim))
        self.cache = {}

    def forward(self, X):
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = np.maximum(0, Z1)  # ReLU
        Z2 = np.dot(A1, self.W2) + self.b2
        
        # Log-Sum-Exp Stable Softmax
        max_Z2 = np.max(Z2, axis=1, keepdims=True)
        exp_Z2 = np.exp(Z2 - max_Z2)
        Y_hat = exp_Z2 / np.sum(exp_Z2, axis=1, keepdims=True)
        
        self.cache = {'X': X, 'Z1': Z1, 'A1': A1, 'Y_hat': Y_hat}
        return Y_hat

    def compute_loss(self, Y_hat, Y_true):
        eps = 1e-15
        return -np.sum(Y_true * np.log(np.clip(Y_hat, eps, 1.0))) / len(Y_true)

    def backward(self, Y_true):
        X = self.cache['X']
        Z1 = self.cache['Z1']
        A1 = self.cache['A1']
        Y_hat = self.cache['Y_hat']
        N = X.shape[0]

        dZ2 = (Y_hat - Y_true) / N
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (Z1 > 0)
        dW1 = np.dot(X.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

# 4. Train with Pure NumPy Mini-Batch Adam Optimizer
model_mnist = ScratchMNISTModel(in_dim=784, hidden_dim=128, out_dim=10)
adam_opt = Adam(lr=0.003, beta1=0.9, beta2=0.999)

batch_size = 128
epochs = 12
n_train = len(X_train_sub)
n_batches = n_train // batch_size

loss_trace = []
train_acc_trace = []

print("=" * 70)
print(f"TRAINING SCRATCH NEURAL NETWORK ON REAL MNIST ({n_train} samples, Adam)")
print("=" * 70)

for epoch in range(epochs):
    # Shuffle dataset
    shuffle_idx = np.random.permutation(n_train)
    X_shuffled = X_train_sub[shuffle_idx]
    Y_shuffled = Y_train_onehot[shuffle_idx]

    epoch_loss = 0.0
    for b in range(n_batches):
        start_i = b * batch_size
        end_i = start_i + batch_size
        X_b = X_shuffled[start_i:end_i]
        Y_b = Y_shuffled[start_i:end_i]

        # Forward
        probs = model_mnist.forward(X_b)
        loss = model_mnist.compute_loss(probs, Y_b)
        epoch_loss += loss

        # Backward
        grads = model_mnist.backward(Y_b)

        # Adam Parameter Updates
        model_mnist.W1 = adam_opt.update(model_mnist.W1, grads['W1'], 'param_0')
        model_mnist.b1 = adam_opt.update(model_mnist.b1, grads['b1'], 'b1')
        model_mnist.W2 = adam_opt.update(model_mnist.W2, grads['W2'], 'W2')
        model_mnist.b2 = adam_opt.update(model_mnist.b2, grads['b2'], 'b2')

    avg_loss = epoch_loss / n_batches
    # Compute epoch training accuracy
    train_preds = np.argmax(model_mnist.forward(X_train_sub[:2000]), axis=1)
    train_acc = np.mean(train_preds == y_train_sub[:2000]) * 100.0

    loss_trace.append(avg_loss)
    train_acc_trace.append(train_acc)
    print(f"Epoch {epoch+1:<2}/{epochs} | Avg Batch Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

# 5. Final Evaluation on 10,000 Unseen Test Images!
test_probs = model_mnist.forward(X_test)
test_preds = np.argmax(test_probs, axis=1)
test_accuracy = np.mean(test_preds == y_test_raw) * 100.0

print("=" * 70)
print(f"FINAL TEST SET EVALUATION ON 10,000 UNSEEN MNIST DIGITS:")
print(f"  Test Set Accuracy: {test_accuracy:.2f}%  [Target: >95.0% achieved! 🎉]")
print("=" * 70)

# 6. Visualization: Convergence Curves and Handwritten Predictions Grid
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Subplot 1: Convergence Curves
axes[0].plot(range(1, epochs + 1), loss_trace, color='#ef4444', lw=2.5, marker='o', label='Cross-Entropy Loss')
axes[0].set_ylabel('Loss', color='#ef4444', fontsize=11)
ax2 = axes[0].twinx()
ax2.plot(range(1, epochs + 1), train_acc_trace, color='#2563eb', lw=2.5, marker='s', linestyle='--', label='Train Accuracy (%)')
ax2.set_ylabel('Accuracy (%)', color='#2563eb', fontsize=11)
axes[0].set_title(f'MNIST Scratch Model Training (Final Test Acc: {test_accuracy:.2f}%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].grid(True, linestyle='--', alpha=0.4)

# Subplot 2: 4x4 Grid of Real MNIST Test Predictions
# Visualizing 16 test digits
grid_img = np.zeros((4 * 28, 4 * 28))
sample_indices = np.random.choice(len(X_test), 16, replace=False)

fig_grid, ax_grid = plt.subplots(4, 4, figsize=(7, 7))
for idx, s_i in enumerate(sample_indices):
    r, c = idx // 4, idx % 4
    img = x_test_raw[s_i]
    true_lbl = y_test_raw[s_i]
    pred_lbl = test_preds[s_i]
    
    ax_grid[r, c].imshow(img, cmap='gray')
    color = '#10b981' if true_lbl == pred_lbl else '#ef4444'
    ax_grid[r, c].set_title(f"True:{true_lbl} | Pred:{pred_lbl}", fontsize=9, fontweight='bold', color=color)
    ax_grid[r, c].axis('off')

fig_grid.suptitle('MNIST Test Sample Predictions (Green = Correct)', fontsize=12, fontweight='bold')
plt.tight_layout()
grid_path = os.path.join(assets_dir, 'mnist_scratch_predictions_grid.png')
fig_grid.savefig(grid_path, dpi=120)
plt.close(fig_grid)

# Save loss plot
curve_path = os.path.join(assets_dir, 'mnist_training_convergence.png')
axes[0].figure.savefig(curve_path, dpi=120)
plt.close(axes[0].figure)

print(f"Figures successfully saved:")
print(f"  - {curve_path}")
print(f"  - {grid_path}")


TRAINING SCRATCH NEURAL NETWORK ON REAL MNIST (10000 samples, Adam)
Epoch 1 /12 | Avg Batch Loss: 0.5779 | Train Acc: 91.45%
Epoch 2 /12 | Avg Batch Loss: 0.2444 | Train Acc: 94.70%
Epoch 3 /12 | Avg Batch Loss: 0.1736 | Train Acc: 96.00%
Epoch 4 /12 | Avg Batch Loss: 0.1324 | Train Acc: 97.25%
Epoch 5 /12 | Avg Batch Loss: 0.0965 | Train Acc: 98.20%
Epoch 6 /12 | Avg Batch Loss: 0.0727 | Train Acc: 99.10%
Epoch 7 /12 | Avg Batch Loss: 0.0548 | Train Acc: 98.30%
Epoch 8 /12 | Avg Batch Loss: 0.0430 | Train Acc: 99.70%
Epoch 9 /12 | Avg Batch Loss: 0.0307 | Train Acc: 99.75%
Epoch 10/12 | Avg Batch Loss: 0.0218 | Train Acc: 99.75%
Epoch 11/12 | Avg Batch Loss: 0.0175 | Train Acc: 100.00%
Epoch 12/12 | Avg Batch Loss: 0.0126 | Train Acc: 99.90%
FINAL TEST SET EVALUATION ON 10,000 UNSEEN MNIST DIGITS:
  Test Set Accuracy: 95.34%  [Target: >95.0% achieved! 🎉]
Figures successfully saved:
  - Module-05-Neural-Networks/assets\mnist_training_convergence.png
  - Module-05-Neural-Networks/assets

---

## 📝 Chapter Summary

| Optimizer | Memory Formula | Update Rule | Primary Advantage | Typical Use Case |
|:---|:---|:---|:---|:---|
| **Vanilla SGD** | None | $\theta \leftarrow \theta - \eta \mathbf{g}$ | Simple, zero extra memory | Theoretical baseline |
| **SGD with Momentum** | $\mathbf{v} = \beta \mathbf{v} + \eta \mathbf{g}$ | $\theta \leftarrow \theta - \mathbf{v}$ | Dampens oscillations in steep ravines | Computer Vision (ResNet) |
| **RMSprop** | $\mathbf{s} = \beta \mathbf{s} + (1-\beta)\mathbf{g}^2$ | $\theta \leftarrow \theta - \frac{\eta}{\sqrt{\mathbf{s}}+\epsilon}\mathbf{g}$ | Adapts step size for sparse/noisy features | RNNs, Reinforcement Learning |
| **Adam** | $\mathbf{m}$ (Momentum) + $\mathbf{v}$ (RMSprop) | $\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{\mathbf{v}}}+\epsilon}\hat{\mathbf{m}}$ | Best-in-class general convergence | Universal default (LLMs, Transformers) |
| **Cosine Annealing** | Half-wave decay | $\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\frac{t}{T}\pi))$ | Eliminates sudden step drops, improves test generalization | SOTA training schedules |

---


## 🔗 Resources & Further Reading

| Resource | Type | Why Read It |
|:---|:---|:---|
| **[Adam: A Method for Stochastic Optimization (Kingma & Ba, 2014)](https://arxiv.org/abs/1412.6980)** | Seminal Paper | The landmark paper introducing Adam, now cited over 150,000 times. |
| **[An Overview of Gradient Descent Optimization Algorithms (Sebastian Ruder)](https://ruder.io/optimizing-gradient-descent/)** | Authoritative Survey | The definitive visual reference comparing SGD, Momentum, NAG, AdaGrad, RMSprop, and Adam. |
| **[SGDR: Stochastic Gradient Descent with Warm Restarts (Loshchilov & Hutter, 2016)](https://arxiv.org/abs/1608.03983)** | Foundational Paper | Introduced Cosine Annealing learning rate schedules and cyclical restarts. |
| **[Decoupled Weight Decay Regularization (AdamW) (Loshchilov & Hutter, 2017)](https://arxiv.org/abs/1711.05101)** | Modern Paper | Explains why standard L2 regularization fails in Adam and formalizes modern AdamW used in Transformers. |

---


## 🏆 Module 05 Capstone Complete — You Built Deep Learning from Scratch!

Congratulations! You have completed **Module 05: Neural Networks from Scratch**.
You did not import PyTorch, Keras, or TensorFlow. You derived the equations, built the forward matrix math, implemented the reverse chain rule, verified your gradients with numerical finite differences, and trained an optimizer from scratch to recognize handwritten digits with $> 95\%$ accuracy!

Now that you possess the deep mathematical intuition of what is happening under the hood of tensor engines, you are finally ready to touch industry-grade frameworks!

---

## ➡️ Next Module: Module 06 — PyTorch from Scratch

Now we transition to the industry-standard framework powering modern AI research:
- **Tensors & GPU Acceleration:** Moving operations from CPU to CUDA tensor cores.
- **PyTorch Autograd Tape:** How PyTorch tracks computation graphs dynamically under the hood.
- **The `nn.Module` Architecture:** Building reusable modular layers and custom models.
- **Dataset & DataLoader:** High-performance multi-threaded data streaming pipelines.
- **The Canonical PyTorch Training Loop:** Zeroing gradients, backward tape execution, and optimizer steps.

---
*Made with ❤️ | Zero to Hero — AI Engineer Learning Path*
